In [0]:
import pyspark.sql.functions as F
from pyspark.sql import SparkSession
from datetime import datetime
import pytz

In [0]:
%run ../../functions/functions

In [0]:
fuso_br = pytz.timezone('America/Sao_Paulo')
agora = datetime.now(fuso_br)
particao_hoje = f"ano={agora.year}/mes={agora.month:02d}/dia={agora.day:02d}"

In [0]:
storage_origem = "landingbeca2026jan"

spark.conf.set(
    f"fs.azure.account.key.{storage_origem}.dfs.core.windows.net",
    storage_key_landing
)

meu_storage = "stgbbb"
meu_container = "landing"

spark.conf.set(
    f"fs.azure.account.key.{meu_storage}.dfs.core.windows.net",
    storage_key
)

In [0]:
containers_origem = ["balancacomercial", "cnpj"]
caminho_destino = f"abfss://{meu_container}@{meu_storage}.dfs.core.windows.net/"

In [0]:
for container_atual in containers_origem:    
    caminho_origem = f"abfss://{container_atual}@landingbeca2026jan.dfs.core.windows.net/"
    try:
        arquivos = dbutils.fs.ls(caminho_origem)
        
        for arquivo in arquivos:
            if arquivo.isFile():
                nome_arquivo = arquivo.name
                destino_final = f"{caminho_destino}{container_atual}/{particao_hoje}/{nome_arquivo}"
                dbutils.fs.cp(arquivo.path, destino_final)
                # print(destino_final)
    
    except Exception as e:
        print(e)